In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
import os

In [5]:
load_dotenv()

True

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI


model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", google_api_key=os.getenv("GEMINI_API_KEY"))

In [7]:
# create a state graph object
rating = []

class BlogState(TypedDict):
    
    title: str
    outline: str
    content: str
    rate: int


In [8]:
# function to create an outline for a blog post based on the title

def create_outline(state: BlogState)-> BlogState:
    
    # fetch titile
    title = state['title']
    
    # form a prompt for the LLM
    prompt = f'Create a detailed outline for a blog post with the title: {title}'
  
    outline = model.invoke(prompt).content
    
    #update the state with the outline
    state['outline'] = outline
    
    return state

In [9]:
# function to create content for a blog post based on the outline

def create_blog(state: BlogState) -> BlogState:
    
    # fetch information from the state
    title = state['title']
    outline = state['outline']
    
    # form a prompt for the LLM
    prompt = f'Create a detailed blog post based on the following title and outline.\nTitle: {title}\nOutline: {outline}'
    
    # ask the LLM to answer the question
    content = model.invoke(prompt).content
    
    #update state 
    state['content'] = content
    return state

In [10]:
# funtion for rating the blog content

def rate_content(state: BlogState) -> BlogState:
    
    title = state['title']
    outline = state['outline']
    content = state['content']
    
    
    prompt = f'Bases on the title "{title}" and outline "{outline}", rate the following blog content on a scale of 1 to 10, where 1 is very bad and 10 is excellent. Provide only the rating number.\nContent: {content}'
    
    rating = model.invoke(prompt).content
    
    # state['rate'] = int(rating.text)
    print('rating---:', rating)    
    return state 

In [11]:
# create graph

graph = StateGraph(BlogState)

# add nodes

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('rate_content', rate_content) # dummy node to rate the content

# add edges 
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'rate_content')
graph.add_edge('rate_content', END)

# compile graph

workflow = graph.compile()

In [12]:
# execute graph
inial_state = {'title': 'Small LinkedIn post about RAG'}    

final_state = workflow.invoke(inial_state)

print(final_state)
 

rating---: [{'type': 'text', 'text': '10', 'extras': {'signature': 'EokMCoYMAb4+9vvRJr5epZj+0aORdU3eFxDLkPNG4vPArRZ/yfNrzwvQy0GUFPQAF20gfAZlbrGuKHmGRu6LnnDVS/liXsa0ALIdEBakT50UdIkRQ/updzkUH5ZIKI0ncNGuhBw6malJp0CKIZMJqGFyDzWKlgCMahbqE17nWaW61qF6joOxh/cq6VypQYhcZB3b0MKzBKzMqsLpfW5PQMR9hRCbD4gJZozljVqFSn/EUi4ghxiQ548Nsg8MzUN60hxBjb5Y6G2R2tEWrU5XQA4K84Z6WUlUM+GcMg8yapUQO4C4eB5FQ7QjxE9U1Sj/FIMI8fuV76LfXYA5/clvQOMvAuWDuYWM75AyUqb2ni2KCpOEzDwBloEaEFTEjbgj/nBqN/RsEryN47HJjniDwQcoJxWeof6O3ZNS9vQUIITip+PI62b1BfyjDqIydjoCEaWKBL4CFXIkfie/Z0IpCPKArUwx8Up15scn8IRCCtA3QBVsLHkbYV/0uR8+kvssDojcPrS2CJ+uyigCeUDr7YAWKNvsnP0zVAkQ8AA4NBT3DrS/AXur5mbc8wAu8OYMiJfe4U/1tx1OlKhLpDSCoLGWkfvZV2rSbuZHsDl26m5aG3OJM1DEXoxKx2OP5GhBeLeJnbM2c+QwHzvvsBLfU35HlcbjvGYn+H7TJ4Yq94a/GHnuB1z5VqTHHC+bi7YSK3OQbBwrM4ArETTKjg6t3ZicbwkA/GudXm/pzdp11TdjtPdwQsLgtBIjfAW19vyvrEEidYWbBHG/1lxewxygJDLWpu3/UswYO0iC5febnBfOH92BCTPTAZVWWdHndUXC/vUu69lMoMIpkJ7x6/Rc02eCknUjwjvy9ts97EdXmoTthcmsXx1b6+a0/iVbdoS7s7UffLCp97LRGR7iraUorQQrrBOqPHLVecI4